# 09 — Ordinal ensemble: CatBoost + XGBoost + ExtraTrees + Ridge

Blend continuous `sii` predictions before converting them to classes. CatBoost and XGBoost are the strongest existing models; ExtraTrees and Ridge provide different inductive biases. We compare each member, equal averaging, and a nonnegative QWK-tuned blend, including single-model candidates so that blending is not assumed to help.

Repository handoff: notebooks 01–05 audit raw data; 06 produces row-local Layer A features; 07/08 train individual models. This notebook reads `data/processed/*_features.parquet`, reuses `src/imputation.py` and the ID-stable folds from `src/evaluation.py`, and saves diagnostics plus a submission under `results/`. No time-series data, PCIAT leakage, pseudo-labels, or Kaggle API calls.

**Evaluation:** five outer folds × three inner folds. Inner OOF labels alone determine weights and thresholds for each outer validation fold. Early stopping uses a separate training-only holdout, followed by refitting on the entire training partition. The prior boosting parameters were selected in notebooks 07/08 using this dataset: this is nested *ensemble calibration*, not a fully untouched estimate of all historical model selection. Full-OOF tuned scores are explicitly marked exploratory and cannot be compared to nested scores as if they shared a protocol.

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.config import ID_COLUMN, LEAKAGE_PREFIX, PROCESSED_DIR, PROJECT_ROOT, RANDOM_STATE, RESULTS_DIR, TARGET
from src.ensemble import DEFAULT_THRESHOLDS, fit_blend, optimize_thresholds, to_classes
from src.evaluation import create_cv_splits
from src.imputation import make_preprocessor

MODEL_NAMES = ["catboost", "xgboost_extra", "extra_trees", "ridge"]
INNER_SPLITS = 3
WEIGHT_RESOLUTION = 4  # 0.25 increments, 35 simplex candidates; raise only with fresh validation.
THREADS = min(4, os.cpu_count() or 1)
MAX_TREES = 3000
STOPPING_ROUNDS = 100

def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, labels=[0, 1, 2, 3], weights="quadratic")

print("Models:", MODEL_NAMES, "| CPU threads:", THREADS)

Models: ['catboost', 'xgboost_extra', 'extra_trees', 'ridge'] | CPU threads: 4


## 1. Load the same labeled participants and fold assignment

Only deterministic row-local transformations happen before CV. Missing categorical values become a literal level; all learned medians, scaling, and one-hot vocabularies remain inside model fitting.

In [2]:
train_path = PROCESSED_DIR / "train_features.parquet"
test_path = PROCESSED_DIR / "test_features.parquet"
if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError("Run notebooks/06_imputation_strategy.ipynb first.")

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)
labeled = train_features.loc[train_features[TARGET].notna()].reset_index(drop=True)
feature_columns = [
    c for c in labeled.columns
    if c not in {ID_COLUMN, TARGET} and not c.startswith(LEAKAGE_PREFIX)
]
missing_columns = set(feature_columns) - set(test_features.columns)
if missing_columns:
    raise ValueError(f"Missing processed test features: {sorted(missing_columns)}")
X = labeled[feature_columns].copy()
X_test = test_features[feature_columns].copy()
y = labeled[TARGET].astype(int)
ids = labeled[ID_COLUMN]
assert ids.notna().all() and ids.is_unique
assert y.isin([0, 1, 2, 3]).all()

categorical_columns = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
for frame in (X, X_test):
    for column in categorical_columns:
        frame[column] = frame[column].astype("object").where(frame[column].notna(), "__missing__").astype(str)
    numeric = frame.select_dtypes(include="number").columns
    frame[numeric] = frame[numeric].replace([np.inf, -np.inf], np.nan)

def add_extra_features(frame):
    # Same five experimental features as notebook 08; no learned statistics.
    extra = frame.copy()
    extra["Fitness_Endurance_total_seconds"] = extra["Fitness_Endurance-Time_Mins"] * 60 + extra["Fitness_Endurance-Time_Sec"]
    extra["Physical-Waist_to_Height"] = extra["Physical-Waist_Circumference"] / extra["Physical-Height"].replace(0, np.nan)
    extra["Physical-Pulse_Pressure"] = extra["Physical-Systolic_BP"] - extra["Physical-Diastolic_BP"]
    extra["FGC_total_score"] = extra[[
        "FGC-FGC_CU", "FGC-FGC_GSND", "FGC-FGC_GSD", "FGC-FGC_PU",
        "FGC-FGC_SRL", "FGC-FGC_SRR", "FGC-FGC_TL",
    ]].sum(axis=1, skipna=True)  # Matches notebook 08, including zero for an absent block.
    extra["BMI_per_Age"] = extra["Physical-BMI"] / extra["Basic_Demos-Age"].replace(0, np.nan)
    return extra.replace([np.inf, -np.inf], np.nan)

X_extra, X_test_extra = add_extra_features(X), add_extra_features(X_test)
cv_splits = create_cv_splits(X=X, y=y, ids=ids)
print("Labeled:", X.shape, "| test:", X_test.shape)
print("Excluded unlabeled rows:", len(train_features) - len(labeled))
print("Class counts:", y.value_counts().sort_index().to_dict())
print("Outer validation sizes:", [len(val) for _, val in cv_splits])

Labeled: (2736, 65) | test: (20, 65)
Excluded unlabeled rows: 1224
Class counts: {0: 1594, 1: 730, 2: 378, 3: 34}
Outer validation sizes: [548, 547, 547, 547, 547]


## 2. Frozen member configurations and training-only early stopping

Boosting parameters are copied from the saved best trials in 07/08, without rerunning Optuna. ExtraTrees uses shallow, leaf-regularized randomized trees; Ridge uses the project's median-imputation pipeline.

For every inner/outer fit, boosting first selects its tree count on an 85/15 split of the available training rows. Its preprocessor is fit on the 85% only. Then a fresh model (and fresh preprocessing) is refit on all available training rows with that fixed tree count. Outer validation labels never enter this procedure.

In [3]:
CATBOOST_PARAMS = {
    "learning_rate": 0.010188851583434083, "depth": 7,
    "l2_leaf_reg": 16.361624028536067,
    "bagging_temperature": 2.482872677273449,
    "random_strength": 1.4065544206797491,
}
XGBOOST_PARAMS = {
    "learning_rate": 0.014850259678316026, "max_depth": 5,
    "min_child_weight": 1.413347955356425,
    "subsample": 0.8762511234570196, "colsample_bytree": 0.5605936678257406,
    "reg_alpha": 3.726158311196111, "reg_lambda": 23.07622742870599,
    "gamma": 1.7853520582128581,
}

def feature_frame(name, base, extra):
    return extra if name == "xgboost_extra" else base

def build_booster(name, frame, seed, tree_count, early_stopping=False):
    if name == "catboost":
        params = dict(
            loss_function="RMSE", iterations=tree_count,
            cat_features=tuple(frame.columns.get_loc(c) for c in categorical_columns),
            random_seed=seed, thread_count=THREADS, verbose=False,
            allow_writing_files=False, **CATBOOST_PARAMS,
        )
        if early_stopping:
            params["early_stopping_rounds"] = STOPPING_ROUNDS
        return CatBoostRegressor(**params)
    params = dict(
        objective="reg:squarederror", eval_metric="rmse",
        n_estimators=tree_count, random_state=seed, n_jobs=THREADS,
        tree_method="hist", **XGBOOST_PARAMS,
    )
    if early_stopping:
        params["early_stopping_rounds"] = STOPPING_ROUNDS
    return XGBRegressor(**params)

def fit_member(name, frame, labels, seed):
    if name in {"ridge", "extra_trees"}:
        estimator = Ridge(alpha=1.0) if name == "ridge" else ExtraTreesRegressor(
            n_estimators=400, max_depth=12, min_samples_leaf=8,
            max_features=0.8, random_state=seed, n_jobs=THREADS,
        )
        model = Pipeline([
            ("preprocessor", make_preprocessor(frame, strategy="median", scale=name == "ridge")),
            ("model", estimator),
        ])
        model.fit(frame, labels)
        return model, None

    fit_pos, stop_pos = train_test_split(
        np.arange(len(labels)), test_size=0.15, random_state=seed, stratify=labels,
    )
    fit_frame, stop_frame = frame.iloc[fit_pos], frame.iloc[stop_pos]
    provisional = build_booster(name, frame, seed, MAX_TREES, early_stopping=True)
    if name == "catboost":
        provisional.fit(fit_frame, labels.iloc[fit_pos],
                        eval_set=(stop_frame, labels.iloc[stop_pos]), use_best_model=True)
        tree_count = max(1, provisional.tree_count_)
        final_model = build_booster(name, frame, seed, tree_count)
    else:
        preprocessor = make_preprocessor(fit_frame, strategy="median", scale=False)
        fit_values = preprocessor.fit_transform(fit_frame)
        stop_values = preprocessor.transform(stop_frame)
        provisional.fit(fit_values, labels.iloc[fit_pos],
                        eval_set=[(stop_values, labels.iloc[stop_pos])], verbose=False)
        tree_count = provisional.best_iteration + 1
        final_model = Pipeline([
            ("preprocessor", make_preprocessor(frame, strategy="median", scale=False)),
            ("model", build_booster(name, frame, seed, tree_count)),
        ])
    final_model.fit(frame, labels)
    return final_model, int(tree_count)

def inner_splits(frame, labels, participant_ids):
    # ID-stable, just like the outer folds, but only within the outer training set.
    sorted_positions = np.argsort(participant_ids.astype(str).to_numpy())
    cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [
        (sorted_positions[fit], sorted_positions[val])
        for fit, val in cv.split(frame.iloc[sorted_positions], labels.iloc[sorted_positions])
    ]

def collect_inner_oof(base, extra, labels, participant_ids):
    predictions = np.full((len(labels), len(MODEL_NAMES)), np.nan)
    for inner_fold, (fit_idx, val_idx) in enumerate(inner_splits(base, labels, participant_ids), start=1):
        for column, name in enumerate(MODEL_NAMES):
            frame = feature_frame(name, base, extra)
            model, _ = fit_member(name, frame.iloc[fit_idx], labels.iloc[fit_idx], RANDOM_STATE)
            predictions[val_idx, column] = np.asarray(model.predict(frame.iloc[val_idx])).ravel()
        print(f"  inner fold {inner_fold}/{INNER_SPLITS} complete", flush=True)
    assert np.isfinite(predictions).all()
    return predictions

## 3. Nested calibration and outer OOF predictions

Fit each member's thresholds, equal-blend thresholds, and optimized-blend weights/thresholds on the inner OOF predictions only. Score the untouched outer fold once. Save the predictions and each fold's calibration parameters for auditing.

In [4]:
oof_members = np.full((len(y), len(MODEL_NAMES)), np.nan)
fold_assignment = np.full(len(y), -1, dtype=int)
nested_predictions = {
    **{f"{name} | nested thresholds": np.full(len(y), -1, dtype=int) for name in MODEL_NAMES},
    "equal blend | fixed thresholds": np.full(len(y), -1, dtype=int),
    "equal blend | nested thresholds": np.full(len(y), -1, dtype=int),
    "optimized blend | nested calibration": np.full(len(y), -1, dtype=int),
}
fold_rows, calibration_log = [], []
started = time.monotonic()

for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
    print(f"Outer fold {fold}/{len(cv_splits)}", flush=True)
    base_train, extra_train = X.iloc[train_idx], X_extra.iloc[train_idx]
    labels_train = y.iloc[train_idx]
    inner_oof = collect_inner_oof(base_train, extra_train, labels_train, ids.iloc[train_idx])
    member_thresholds = {
        name: optimize_thresholds(inner_oof[:, column], labels_train)
        for column, name in enumerate(MODEL_NAMES)
    }
    equal_thresholds = optimize_thresholds(inner_oof.mean(axis=1), labels_train)
    blend = fit_blend(inner_oof, labels_train, resolution=WEIGHT_RESOLUTION)

    tree_counts = {}
    for column, name in enumerate(MODEL_NAMES):
        frame = feature_frame(name, X, X_extra)
        model, tree_count = fit_member(name, frame.iloc[train_idx], labels_train, RANDOM_STATE)
        oof_members[val_idx, column] = np.asarray(model.predict(frame.iloc[val_idx])).ravel()
        tree_counts[name] = tree_count
        nested_predictions[f"{name} | nested thresholds"][val_idx] = to_classes(
            oof_members[val_idx, column], member_thresholds[name],
        )
    fold_assignment[val_idx] = fold
    equal_scores = oof_members[val_idx].mean(axis=1)
    nested_predictions["equal blend | fixed thresholds"][val_idx] = to_classes(equal_scores)
    nested_predictions["equal blend | nested thresholds"][val_idx] = to_classes(equal_scores, equal_thresholds)
    nested_predictions["optimized blend | nested calibration"][val_idx] = to_classes(
        oof_members[val_idx] @ blend["weights"], blend["thresholds"],
    )
    for setup, predictions in nested_predictions.items():
        score = qwk(y.iloc[val_idx], predictions[val_idx])
        fold_rows.append({"setup": setup, "fold": fold, "validation_qwk": score})
    calibration_log.append({
        "fold": fold,
        "weights": dict(zip(MODEL_NAMES, blend["weights"].tolist())),
        "thresholds": blend["thresholds"].tolist(),
        "inner_calibration_qwk": blend["calibration_qwk"],
        "member_thresholds": {name: thresholds.tolist() for name, thresholds in member_thresholds.items()},
        "equal_thresholds": equal_thresholds.tolist(),
        "tree_counts": tree_counts,
    })
    print("  blend weights:", dict(zip(MODEL_NAMES, blend["weights"].round(2))))
    print(f"  outer blend QWK: {fold_rows[-1]['validation_qwk']:.4f}; elapsed {(time.monotonic()-started)/60:.1f} min", flush=True)

assert np.isfinite(oof_members).all()
assert (fold_assignment > 0).all()
assert all(np.isin(predictions, [0, 1, 2, 3]).all() for predictions in nested_predictions.values())

Outer fold 1/5


  inner fold 1/3 complete


  inner fold 2/3 complete


  inner fold 3/3 complete


  blend weights: {'catboost': np.float64(0.25), 'xgboost_extra': np.float64(0.75), 'extra_trees': np.float64(0.0), 'ridge': np.float64(0.0)}
  outer blend QWK: 0.4642; elapsed 1.2 min


...

Outer fold 5/5


  inner fold 1/3 complete


  inner fold 2/3 complete


  inner fold 3/3 complete


  blend weights: {'catboost': np.float64(0.5), 'xgboost_extra': np.float64(0.0), 'extra_trees': np.float64(0.25), 'ridge': np.float64(0.25)}
  outer blend QWK: 0.4185; elapsed 5.1 min


In [5]:
fold_results = pd.DataFrame(fold_rows)
summary_rows = []
for setup, predictions in nested_predictions.items():
    scores = fold_results.loc[fold_results["setup"].eq(setup), "validation_qwk"].to_numpy()
    summary_rows.append({
        "setup": setup, "protocol": "outer validation; calibration on inner OOF only",
        "oof_qwk": qwk(y, predictions), "mean_val_qwk": scores.mean(),
        "val_std_qwk": scores.std(),
        **{f"fold_{fold}_qwk": score for fold, score in enumerate(scores, start=1)},
        **{f"pred_{class_id}_count": int((predictions == class_id).sum()) for class_id in range(4)},
    })
nested_summary = pd.DataFrame(summary_rows).sort_values("oof_qwk", ascending=False)
print(nested_summary[["setup", "oof_qwk", "mean_val_qwk", "val_std_qwk"]].round(4).to_string(index=False))

single_rows = nested_summary[nested_summary["setup"].str.startswith(tuple(MODEL_NAMES))]
best_single = single_rows.iloc[0]
blend_qwk = qwk(y, nested_predictions["optimized blend | nested calibration"])
print(f"Nested optimized blend minus best nested member: {blend_qwk - best_single['oof_qwk']:+.4f}")
print("Prediction correlation (high correlation limits the benefit of averaging):")
display(pd.DataFrame(oof_members, columns=MODEL_NAMES).corr().round(3))
display(pd.DataFrame(
    confusion_matrix(y, nested_predictions["optimized blend | nested calibration"], labels=[0, 1, 2, 3]),
    index=[f"true_{i}" for i in range(4)], columns=[f"pred_{i}" for i in range(4)],
))

                               setup  oof_qwk  mean_val_qwk  val_std_qwk
     equal blend | nested thresholds   0.4703        0.4697       0.0376
optimized blend | nested calibration   0.4596        0.4593       0.0364
   xgboost_extra | nested thresholds   0.4554        0.4551       0.0383
     extra_trees | nested thresholds   0.4530        0.4527       0.0297
        catboost | nested thresholds   0.4519        0.4514       0.0321
           ridge | nested thresholds   0.4347        0.4364       0.0341
      equal blend | fixed thresholds   0.3663        0.3663       0.0368
Nested optimized blend minus best nested member: +0.0042
Prediction correlation (high correlation limits the benefit of averaging):


,catboost,xgboost_extra,extra_trees,ridge
catboost,1.000,0.965,0.936,0.559
xgboost_extra,0.965,1.000,0.949,0.554
extra_trees,0.936,0.949,1.000,0.541
ridge,0.559,0.554,0.541,1.000


,pred_0,pred_1,pred_2,pred_3
true_0,1128,344,113,9
true_1,276,272,171,11
true_2,79,128,155,16
true_3,0,2,25,7


## 4. Full-OOF calibration: exploratory comparison and deployment weights

The next table uses the *same* labels for choosing thresholds/weights and computing QWK. It is optimistic, like the saved regressor results from 07/08; it is not the nested estimate above. The weight grid includes each member, so its calibration score cannot be worse than its best member calibrated by the same optimizer.

The historical 0.4767 XGBoost result is context only: preprocessing versions, missing-category representation, and the training-partition refit differ here. Always use the newly rerun members for the direct comparison.

In [6]:
deployment_blend = fit_blend(oof_members, y, resolution=WEIGHT_RESOLUTION)
exploratory_rows = []
for column, name in enumerate(MODEL_NAMES):
    thresholds = optimize_thresholds(oof_members[:, column], y)
    exploratory_rows.append({
        "setup": name, "protocol": "full OOF calibration; optimistic",
        "oof_qwk": qwk(y, to_classes(oof_members[:, column], thresholds)),
    })
equal_thresholds_full = optimize_thresholds(oof_members.mean(axis=1), y)
exploratory_rows.append({
    "setup": "equal blend", "protocol": "full OOF calibration; optimistic",
    "oof_qwk": qwk(y, to_classes(oof_members.mean(axis=1), equal_thresholds_full)),
})
exploratory_rows.append({
    "setup": "optimized blend", "protocol": "full OOF calibration; optimistic",
    "oof_qwk": deployment_blend["calibration_qwk"],
})
exploratory_summary = pd.DataFrame(exploratory_rows).sort_values("oof_qwk", ascending=False)
print(exploratory_summary.round(4).to_string(index=False))
print("Deployment weights:", dict(zip(MODEL_NAMES, deployment_blend["weights"].round(3))))
print("Deployment thresholds:", deployment_blend["thresholds"].round(4))
print(f"Exploratory delta vs historical 0.4767: {deployment_blend['calibration_qwk'] - 0.4767:+.4f}")
if np.count_nonzero(deployment_blend["weights"]) == 1:
    print("The grid selected a single model: no full-OOF ensemble gain.")
if blend_qwk <= best_single["oof_qwk"]:
    print("Nested optimized blending did not beat the best nested member. Do not claim a validated gain.")

          setup                         protocol  oof_qwk
optimized blend full OOF calibration; optimistic   0.4782
  xgboost_extra full OOF calibration; optimistic   0.4749
    equal blend full OOF calibration; optimistic   0.4727
    extra_trees full OOF calibration; optimistic   0.4677
       catboost full OOF calibration; optimistic   0.4648
          ridge full OOF calibration; optimistic   0.4486
Deployment weights: {'catboost': np.float64(0.25), 'xgboost_extra': np.float64(0.5), 'extra_trees': np.float64(0.0), 'ridge': np.float64(0.25)}
Deployment thresholds: [0.5537 0.9113 1.449 ]
Exploratory delta vs historical 0.4767: +0.0015


## 5. Refit deployment members and produce a submission

A fresh fit on all labeled rows gives predictions on processed test features. Only members with nonzero deployment weights need to be trained. The sample submission defines ID order; do not assume the parquet order matches it. The provided competition test CSV may be a small public example, not the hidden leaderboard test.

In [7]:
test_members = np.zeros((len(X_test), len(MODEL_NAMES)), dtype=float)
deployment_tree_counts = {}
for column, name in enumerate(MODEL_NAMES):
    if deployment_blend["weights"][column] == 0:
        continue
    train_frame = feature_frame(name, X, X_extra)
    test_frame = feature_frame(name, X_test, X_test_extra)
    model, tree_count = fit_member(name, train_frame, y, RANDOM_STATE)
    test_members[:, column] = np.asarray(model.predict(test_frame)).ravel()
    deployment_tree_counts[name] = tree_count
    print(f"Refit {name}: trees={tree_count}", flush=True)

test_continuous = test_members @ deployment_blend["weights"]
test_classes = to_classes(test_continuous, deployment_blend["thresholds"])
sample_submission = pd.read_csv(PROJECT_ROOT / "data" / "sample_submission.csv")
test_ids = test_features[ID_COLUMN]
assert test_ids.notna().all() and test_ids.is_unique
assert sample_submission[ID_COLUMN].notna().all() and sample_submission[ID_COLUMN].is_unique
if set(sample_submission[ID_COLUMN]) != set(test_ids):
    raise ValueError("sample_submission IDs and processed test IDs do not match")
prediction_by_id = pd.Series(test_classes, index=test_ids.to_numpy())
submission = sample_submission.copy()
submission[TARGET] = submission[ID_COLUMN].map(prediction_by_id).astype(int)
assert submission[TARGET].isin([0, 1, 2, 3]).all()
display(submission.head())

Refit catboost: trees=650


Refit xgboost_extra: trees=245


Refit ridge: trees=None


,id,sii
0,00008ff9,0
1,000fd460,0
2,00105258,0
3,00115b9f,0
4,0016bb22,1


## 6. Save results, predictions, and reproducibility metadata

`ensemble_cv_results.csv` is the nested comparison; `ensemble_exploratory_results.csv` is kept separate. Continuous OOF predictions are tied to participant IDs and fold IDs. The JSON records configurations, package versions, fold calibration and final deployment parameters. Existing model results are not overwritten.

In [8]:
import catboost
import sklearn
import xgboost

RESULTS_DIR.mkdir(exist_ok=True)
nested_summary.to_csv(RESULTS_DIR / "ensemble_cv_results.csv", index=False)
fold_results.to_csv(RESULTS_DIR / "ensemble_fold_results.csv", index=False)
exploratory_summary.to_csv(RESULTS_DIR / "ensemble_exploratory_results.csv", index=False)
oof_table = pd.DataFrame({ID_COLUMN: ids, TARGET: y, "fold": fold_assignment})
for column, name in enumerate(MODEL_NAMES):
    oof_table[f"{name}_continuous"] = oof_members[:, column]
for setup, predictions in nested_predictions.items():
    oof_table[setup] = predictions
oof_table.to_csv(RESULTS_DIR / "ensemble_oof_predictions.csv", index=False)
submission.to_csv(RESULTS_DIR / "submission_ensemble.csv", index=False)
metadata = {
    "seed": RANDOM_STATE, "outer_splits": len(cv_splits), "inner_splits": INNER_SPLITS,
    "weight_resolution": WEIGHT_RESOLUTION, "model_names": MODEL_NAMES,
    "catboost_params": CATBOOST_PARAMS, "xgboost_params": XGBOOST_PARAMS,
    "max_trees": MAX_TREES, "stopping_rounds": STOPPING_ROUNDS,
    "extra_trees_params": {"n_estimators": 400, "max_depth": 12, "min_samples_leaf": 8, "max_features": 0.8},
    "ridge_alpha": 1.0, "feature_columns": feature_columns,
    "versions": {"python": sys.version, "numpy": np.__version__, "pandas": pd.__version__,
                 "sklearn": sklearn.__version__, "catboost": catboost.__version__, "xgboost": xgboost.__version__},
    "evaluation_caveat": "Nested ensemble calibration only; base boosting parameters were previously selected on this dataset.",
    "fold_calibration": calibration_log,
    "deployment": {
        "weights": dict(zip(MODEL_NAMES, deployment_blend["weights"].tolist())),
        "thresholds": deployment_blend["thresholds"].tolist(),
        "full_oof_calibration_qwk_optimistic": deployment_blend["calibration_qwk"],
        "tree_counts": deployment_tree_counts,
    },
}
(RESULTS_DIR / "ensemble_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")
print("Saved ensemble diagnostics and submission to:", RESULTS_DIR)
print(f"Nested blend OOF QWK: {blend_qwk:.4f}")
print(f"Full-OOF calibrated blend QWK (optimistic): {deployment_blend['calibration_qwk']:.4f}")

Saved ensemble diagnostics and submission to: /home/m1r0/Innopolis/F26/PMLDL/proj/predict-internet-usage-ivanov-secret/results
Nested blend OOF QWK: 0.4596
Full-OOF calibrated blend QWK (optimistic): 0.4782
